# BFS Ground Truth Data Generation

This notebook generates ground truth simulation data for Backward Facing Step (BFS) flow at various Reynolds numbers and mesh resolutions. This data will be used to train ML models for super-resolution.

**Workflow:**
1. Set up parameters (Re range, mesh sizes, geometry, convergence criteria)
2. Run BFS simulations for each combination of Re and mesh size
3. Save results to HDF5 files (individual and combined)

In [ ]:
import os

# Create output directory
output_dir = "results"
os.makedirs(output_dir, exist_ok=True)

print(f"Output directory created: {output_dir}")

## Import Required Libraries

In [ ]:
import numpy as np
from numba import njit, prange
import time
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import Dict, Tuple, Optional
import h5py

## Define Classes and Helper Functions for BFS Simulation

In [ ]:
@dataclass
class BoundaryCondition:
    """Class to define boundary conditions"""
    type: str  # 'dirichlet' or 'neumann'
    value: float = 0.0

class BoundaryConditions:
    """Container for all boundary conditions for BFS"""
    def __init__(self):
        # Default BFS boundary conditions
        self.u_boundaries = {
            'left': BoundaryCondition('dirichlet', 0.0),   # Inlet (will be overridden by parabolic profile)
            'right': BoundaryCondition('neumann', 0.0),    # Outlet
            'top': BoundaryCondition('dirichlet', 0.0),    # No-slip wall
            'bottom': BoundaryCondition('dirichlet', 0.0)  # No-slip wall
        }
        self.v_boundaries = {
            'left': BoundaryCondition('dirichlet', 0.0),
            'right': BoundaryCondition('neumann', 0.0),
            'top': BoundaryCondition('dirichlet', 0.0),
            'bottom': BoundaryCondition('dirichlet', 0.0)
        }
        self.p_boundaries = {
            'left': BoundaryCondition('neumann', 0.0),
            'right': BoundaryCondition('dirichlet', 0.0),  # Reference pressure at outlet
            'top': BoundaryCondition('neumann', 0.0),
            'bottom': BoundaryCondition('neumann', 0.0)
        }

class MeshParameters:
    """Class to handle mesh parameters"""
    def __init__(self, nx: int = 100, ny: int = 100, lx: float = 10.0, ly: float = 3.0):
        self.nx = nx
        self.ny = ny
        self.lx = lx
        self.ly = ly
        self.dx = lx / nx
        self.dy = ly / ny
        self.volp = self.dx * self.dy

class FluidProperties:
    """Class to handle fluid properties"""
    def __init__(self, Re: float = 100.0, rho: float = 1.0):
        self.Re = Re
        self.rho = rho
        # For BFS with characteristic length = step height and bulk velocity Ub
        self.nu = 1.0 / Re  # kinematic viscosity

class SolverSettings:
    """Class to handle solver settings"""
    def __init__(self, dt: float = 0.001, max_iterations: int = 100000,
                 convergence_criteria: Dict[str, float] = None,
                 scheme: str = 'UPWIND',
                 relaxation_factors: Dict[str, float] = None):
        self.dt = dt
        self.max_iterations = max_iterations
        self.scheme = scheme
        
        if convergence_criteria is None:
            self.convergence_criteria = {
                'u': 1e-6,
                'v': 1e-6,
                'p': 1e-6,
                'continuity': 1e-6
            }
        else:
            self.convergence_criteria = convergence_criteria
        
        if relaxation_factors is None:
            self.relaxation_factors = {
                'u': 0.7,
                'v': 0.7,
                'p': 0.3
            }
        else:
            self.relaxation_factors = relaxation_factors

## Numba-Compiled Functions for Performance

In [ ]:
@njit
def copy_new_to_old(Var, VarOld, nVar, Nx, Ny):
    for k in range(nVar):
        for i in range(Nx + 2):
            for j in range(Ny + 2):
                VarOld[k, i, j] = Var[k, i, j]

@njit
def apply_bc_configured(Var, k, Nx, Ny, bc_types, bc_values):
    """Apply boundary conditions based on configuration"""
    # Left and Right boundaries
    for j in range(1, Ny + 1):
        if bc_types[0] == 0:  # Dirichlet
            Var[k, 0, j] = 2 * bc_values[0] - Var[k, 1, j]
        else:  # Neumann
            Var[k, 0, j] = Var[k, 1, j]
        
        if bc_types[1] == 0:  # Dirichlet
            Var[k, Nx + 1, j] = 2 * bc_values[1] - Var[k, Nx, j]
        else:  # Neumann
            Var[k, Nx + 1, j] = Var[k, Nx, j]
    
    # Top and Bottom boundaries
    for i in range(1, Nx + 1):
        if bc_types[2] == 0:  # Dirichlet
            Var[k, i, Ny + 1] = 2 * bc_values[2] - Var[k, i, Ny]
        else:  # Neumann
            Var[k, i, Ny + 1] = Var[k, i, Ny]
        
        if bc_types[3] == 0:  # Dirichlet
            Var[k, i, 0] = 2 * bc_values[3] - Var[k, i, 1]
        else:  # Neumann
            Var[k, i, 0] = Var[k, i, 1]

@njit
def linear_interpolation(Var, Ff, Nx, Ny, dx, dy):
    for i in range(1, Nx + 1):
        for j in range(1, Ny + 1):
            Ff[0, i, j] = (Var[0, i, j] + Var[0, i + 1, j]) * dy * 0.5
            Ff[1, i, j] = (Var[1, i, j] + Var[1, i, j + 1]) * dx * 0.5
            Ff[2, i, j] = -(Var[0, i, j] + Var[0, i - 1, j]) * dy * 0.5
            Ff[3, i, j] = -(Var[1, i, j] + Var[1, i, j - 1]) * dx * 0.5

@njit
def simple_upwind(Var, Ff, k, i, j, volp):
    ue, uw, un, us = 0.0, 0.0, 0.0, 0.0
    sum_flux = 0.0
    
    if Ff[0, i, j] >= 0:
        ue = Var[k, i, j]
        sum_flux += Ff[0, i, j]
    else:
        ue = Var[k, i + 1, j]
    
    if Ff[2, i, j] >= 0:
        uw = Var[k, i, j]
        sum_flux += Ff[2, i, j]
    else:
        uw = Var[k, i - 1, j]
    
    if Ff[1, i, j] >= 0:
        un = Var[k, i, j]
        sum_flux += Ff[1, i, j]
    else:
        un = Var[k, i, j + 1]
    
    if Ff[3, i, j] >= 0:
        us = Var[k, i, j]
        sum_flux += Ff[3, i, j]
    else:
        us = Var[k, i, j - 1]
    
    Fc = ue * Ff[0, i, j] + uw * Ff[2, i, j] + un * Ff[1, i, j] + us * Ff[3, i, j]
    ap_c = sum_flux * volp
    
    return Fc, ap_c

@njit
def quick_scheme(Var, Ff, k, i, j, volp):
    ue, uw, un, us = 0.0, 0.0, 0.0, 0.0
    sum_flux = 0.0
    
    # East face
    if Ff[0, i, j] >= 0:
        ue = 0.75 * Var[k, i, j] + 0.375 * Var[k, i + 1, j] - 0.125 * Var[k, i - 1, j]
        sum_flux += 0.75 * Ff[0, i, j]
    else:
        ue = 0.75 * Var[k, i + 1, j] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i + 2, j]
        sum_flux += 0.375 * Ff[0, i, j]
    
    # West face
    if Ff[2, i, j] >= 0:
        uw = 0.75 * Var[k, i, j] + 0.375 * Var[k, i - 1, j] - 0.125 * Var[k, i + 1, j]
        sum_flux += 0.75 * Ff[2, i, j]
    else:
        uw = 0.75 * Var[k, i - 1, j] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i - 2, j]
        sum_flux += 0.375 * Ff[2, i, j]
    
    # North face
    if Ff[1, i, j] >= 0:
        un = 0.75 * Var[k, i, j] + 0.375 * Var[k, i, j + 1] - 0.125 * Var[k, i, j - 1]
        sum_flux += 0.75 * Ff[1, i, j]
    else:
        un = 0.75 * Var[k, i, j + 1] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i, j + 2]
        sum_flux += 0.375 * Ff[1, i, j]
    
    # South face
    if Ff[3, i, j] >= 0:
        us = 0.75 * Var[k, i, j] + 0.375 * Var[k, i, j - 1] - 0.125 * Var[k, i, j + 1]
        sum_flux += 0.75 * Ff[3, i, j]
    else:
        us = 0.75 * Var[k, i, j - 1] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i, j - 2]
        sum_flux += 0.375 * Ff[3, i, j]
    
    Fc = ue * Ff[0, i, j] + uw * Ff[2, i, j] + un * Ff[1, i, j] + us * Ff[3, i, j]
    ap_c = sum_flux * volp
    
    return Fc, ap_c

@njit
def diffusive_flux(Var, k, i, j, dx, dy, volp):
    Fd = volp * ((Var[k, i + 1, j] - 2.0 * Var[k, i, j] + Var[k, i - 1, j]) / (dx * dx) +
                 (Var[k, i, j + 1] - 2.0 * Var[k, i, j] + Var[k, i, j - 1]) / (dy * dy))
    ap_d = -volp * (2.0 / (dx * dx) + 2.0 / (dy * dy))
    return Fd, ap_d

@njit
def update_flux(Var, Ff, dt, rho, Nx, Ny, dx, dy):
    for i in range(1, Nx + 1):
        for j in range(1, Ny + 1):
            Ff[0, i, j] += -dt / rho * (Var[2, i + 1, j] - Var[2, i, j]) * dy / dx
            Ff[1, i, j] += -dt / rho * (Var[2, i, j + 1] - Var[2, i, j]) * dx / dy
            Ff[2, i, j] += -dt / rho * (Var[2, i - 1, j] - Var[2, i, j]) * dy / dx
            Ff[3, i, j] += -dt / rho * (Var[2, i, j - 1] - Var[2, i, j]) * dx / dy

@njit
def under_relax_field(Var, VarOld, k, Nx, Ny, alpha):
    for i in range(1, Nx + 1):
        for j in range(1, Ny + 1):
            Var[k, i, j] = alpha * Var[k, i, j] + (1.0 - alpha) * VarOld[k, i, j]

@njit(parallel=True)
def solve_momentum_upwind(Var, VarOld, Ff, k, Nx, Ny, dx, dy, dt, nu, volp):
    tolerance = 1e-6
    max_iter = 1000
    
    for iter in range(max_iter):
        rms = 0.0
        for i in prange(1, Nx + 1):
            for j in range(1, Ny + 1):
                Fc, ap_c = simple_upwind(Var, Ff, k, i, j, volp)
                Fd, ap_d = diffusive_flux(Var, k, i, j, dx, dy, volp)
                
                R = -(volp / dt * (Var[k, i, j] - VarOld[k, i, j]) + Fc + (-nu) * Fd)
                ap = volp / dt + ap_c + (-nu) * ap_d
                
                Var[k, i, j] = Var[k, i, j] + R / ap
                rms += R * R
        
        rms = np.sqrt(rms / (Nx * Ny))
        if rms < tolerance:
            break

@njit(parallel=True)
def solve_momentum_quick(Var, VarOld, Ff, k, Nx, Ny, dx, dy, dt, nu, volp):
    tolerance = 1e-6
    max_iter = 1000
    
    for iter in range(max_iter):
        rms = 0.0
        for i in prange(1, Nx + 1):
            for j in range(1, Ny + 1):
                Fc, ap_c = quick_scheme(Var, Ff, k, i, j, volp)
                Fd, ap_d = diffusive_flux(Var, k, i, j, dx, dy, volp)
                
                R = -(volp / dt * (Var[k, i, j] - VarOld[k, i, j]) + Fc + (-nu) * Fd)
                ap = volp / dt + ap_c + (-nu) * ap_d
                
                Var[k, i, j] = Var[k, i, j] + R / ap
                rms += R * R
        
        rms = np.sqrt(rms / (Nx * Ny))
        if rms < tolerance:
            break

@njit(parallel=True)
def solve_pressure(Var, Ff, Nx, Ny, dx, dy, dt, rho, volp):
    tolerance = 1e-6
    max_iter = 1000
    k = 2
    
    for iter in range(max_iter):
        rms = 0.0
        for i in prange(1, Nx + 1):
            for j in range(1, Ny + 1):
                Fd, ap_d = diffusive_flux(Var, k, i, j, dx, dy, volp)
                
                LHS = Fd
                RHS = rho / dt * (Ff[0, i, j] + Ff[1, i, j] + Ff[2, i, j] + Ff[3, i, j])
                R = RHS - LHS
                ap = ap_d
                
                Var[k, i, j] = Var[k, i, j] + R / ap
                rms += R * R
        
        rms = np.sqrt(rms / (Nx * Ny))
        if rms < tolerance:
            break

@njit(parallel=True)
def correct_velocity(Var, VarOld, dt, rho, Nx, Ny, dx, dy):
    res_u = 0.0
    res_v = 0.0
    res_p = 0.0
    
    for i in prange(1, Nx + 1):
        for j in range(1, Ny + 1):
            Var[0, i, j] = Var[0, i, j] - dt / rho * (Var[2, i + 1, j] - Var[2, i - 1, j]) / (2 * dx)
            Var[1, i, j] = Var[1, i, j] - dt / rho * (Var[2, i, j + 1] - Var[2, i, j - 1]) / (2 * dy)
            
            res_u += (Var[0, i, j] - VarOld[0, i, j]) ** 2
            res_v += (Var[1, i, j] - VarOld[1, i, j]) ** 2
            res_p += (Var[2, i, j] - VarOld[2, i, j]) ** 2
    
    return res_u, res_v, res_p

## CFD Solver Class for BFS

In [ ]:
class CFDSolver:
    """CFD Solver for BFS flow"""
    def __init__(self, mesh: MeshParameters, fluid: FluidProperties,
                 solver_settings: SolverSettings, bc: BoundaryConditions,
                 step_height: float = 1.0, h: float = 2.0, Ub: float = 1.0):
        self.mesh = mesh
        self.fluid = fluid
        self.settings = solver_settings
        self.bc = bc
        
        # BFS-specific parameters
        self.case_type = 'BFS'
        self.step_height = step_height
        self.h = h
        self.Ub = Ub
        
        # Solution variables
        self.nVar = 3
        self.Var = np.zeros((self.nVar, mesh.nx + 2, mesh.ny + 2))
        self.VarOld = np.zeros((self.nVar, mesh.nx + 2, mesh.ny + 2))
        self.residual = np.zeros(self.nVar)
        self.Ff = np.zeros((4, mesh.nx + 2, mesh.ny + 2))
        self.residual_history = {'u': [], 'v': [], 'p': []}
        
        self._initialize_fields()
    
    def _get_bc_arrays(self, k: int):
        """Convert boundary condition dictionaries to arrays"""
        if k == 0:
            bc_dict = self.bc.u_boundaries
        elif k == 1:
            bc_dict = self.bc.v_boundaries
        else:
            bc_dict = self.bc.p_boundaries
        
        bc_types = np.array([
            0 if bc_dict['left'].type == 'dirichlet' else 1,
            0 if bc_dict['right'].type == 'dirichlet' else 1,
            0 if bc_dict['top'].type == 'dirichlet' else 1,
            0 if bc_dict['bottom'].type == 'dirichlet' else 1
        ], dtype=np.int32)
        
        bc_values = np.array([
            bc_dict['left'].value,
            bc_dict['right'].value,
            bc_dict['top'].value,
            bc_dict['bottom'].value
        ], dtype=np.float64)
        
        return bc_types, bc_values
    
    def _apply_bfs_inlet(self, k: int):
        """
        Apply BFS inlet/wall mixture on the left boundary.
        - For y < step_height: enforce wall (Dirichlet 0) via ghost cell reflection
        - For y >= step_height: apply parabolic U inlet, V = 0
        Only active when self.case_type == 'BFS'.
        """
        if self.case_type != 'BFS':
            return
        if k not in (0, 1):
            return
        
        ny = self.mesh.ny
        dy = self.mesh.dy
        step_h = self.step_height
        h = self.h
        Ub = self.Ub
        
        for j in range(1, ny + 1):
            # Calculate cell-center y coordinate
            y = (j - 0.5) * dy
            
            if y < step_h:
                # Inlet blocked by the step: no-slip wall at x=0
                # Enforce Dirichlet(0) using ghost reflection
                self.Var[k, 0, j] = -self.Var[k, 1, j]
            else:
                # Open inlet part
                if k == 1:
                    # V = 0 across inlet
                    self.Var[1, 0, j] = -self.Var[1, 1, j]
                else:
                    # Parabolic U profile over height h above the step
                    yprime = y - step_h
                    # Clamp within [0, h]
                    if yprime < 0.0:
                        yprime = 0.0
                    if yprime > h:
                        yprime = h
                    
                    # Parabolic profile: u(y') = 6*Ub*(y'/h)*(1 - y'/h)
                    u_in = 6.0 * Ub * (yprime / h) * (1.0 - (yprime / h))
                    self.Var[0, 0, j] = 2.0 * u_in - self.Var[0, 1, j]
                    
                    # Also ensure V ghost enforces v=0 consistently
                    self.Var[1, 0, j] = -self.Var[1, 1, j]
    
    def _apply_bc_wrapper(self, k: int):
        """Apply boundary conditions"""
        # First apply general boundary conditions
        bc_types, bc_values = self._get_bc_arrays(k)
        apply_bc_configured(self.Var, k, self.mesh.nx, self.mesh.ny, bc_types, bc_values)
        
        # Override left boundary for BFS inlet/wall mix
        self._apply_bfs_inlet(k)
    
    def _initialize_fields(self):
        """Initialize all fields"""
        self.Var.fill(0.0)
        self.VarOld.fill(0.0)
        self.Ff.fill(0.0)
        
        for k in range(self.nVar):
            self._apply_bc_wrapper(k)
        
        copy_new_to_old(self.Var, self.VarOld, self.nVar, self.mesh.nx, self.mesh.ny)
        linear_interpolation(self.Var, self.Ff, self.mesh.nx, self.mesh.ny,
                           self.mesh.dx, self.mesh.dy)
    
    def solve(self, output_base_name: str = "output", verbose: bool = True):
        """Main solver loop"""
        count = 0
        converged = False
        start_time = time.time()
        
        if verbose:
            print(f"Starting BFS simulation: Re={self.fluid.Re}, mesh={self.mesh.nx}x{self.mesh.ny}")
            print(f"Time step: {self.settings.dt}, Scheme: {self.settings.scheme}")
            print(f"BFS Parameters: step_height={self.step_height}, h={self.h}, Ub={self.Ub}")
            print(f"Domain: lx={self.mesh.lx}, ly={self.mesh.ly}")
            print("\nIteration\tU-RMS\t\tV-RMS\t\tP-RMS")
            print("-" * 60)
        
        while not converged and count < self.settings.max_iterations:
            count += 1
            self._implicit_solve()
            
            if verbose and count % 100 == 0:
                print(f"{count}", end="")
            
            converged, rms = self._convergence_check(verbose and count % 100 == 0)
            
            # Store residual history
            self.residual_history['u'].append(rms[0])
            self.residual_history['v'].append(rms[1])
            self.residual_history['p'].append(rms[2])
        
        end_time = time.time()
        
        if verbose:
            print(f"\n\nSimulation completed in {end_time - start_time:.2f} seconds")
            print(f"Total iterations: {count}")
        
        self._save_results(output_base_name)
        
        return count, end_time - start_time
    
    def _implicit_solve(self):
        """Implicit solver step using SIMPLE algorithm with under-relaxation"""
        self.residual.fill(0.0)
        
        # Fetch under-relaxation factors
        alpha_u = self.settings.relaxation_factors.get('u', 0.5)
        alpha_v = self.settings.relaxation_factors.get('v', 0.5)
        alpha_p = self.settings.relaxation_factors.get('p', 0.2)
        
        # Solve momentum equations (U and V)
        for k in range(2):
            if self.settings.scheme == 'QUICK':
                solve_momentum_quick(self.Var, self.VarOld, self.Ff, k, self.mesh.nx,
                                   self.mesh.ny, self.mesh.dx, self.mesh.dy,
                                   self.settings.dt, self.fluid.nu, self.mesh.volp)
            else:  # UPWIND
                solve_momentum_upwind(self.Var, self.VarOld, self.Ff, k, self.mesh.nx,
                                    self.mesh.ny, self.mesh.dx, self.mesh.dy,
                                    self.settings.dt, self.fluid.nu, self.mesh.volp)
            
            # Under-relax U and V
            if k == 0:
                under_relax_field(self.Var, self.VarOld, 0, self.mesh.nx, self.mesh.ny, alpha_u)
            else:
                under_relax_field(self.Var, self.VarOld, 1, self.mesh.nx, self.mesh.ny, alpha_v)
            
            self._apply_bc_wrapper(k)
        
        linear_interpolation(self.Var, self.Ff, self.mesh.nx, self.mesh.ny,
                           self.mesh.dx, self.mesh.dy)
        
        # Solve pressure equation
        solve_pressure(self.Var, self.Ff, self.mesh.nx, self.mesh.ny,
                      self.mesh.dx, self.mesh.dy, self.settings.dt,
                      self.fluid.rho, self.mesh.volp)
        
        # Under-relax pressure before correction
        under_relax_field(self.Var, self.VarOld, 2, self.mesh.nx, self.mesh.ny, alpha_p)
        self._apply_bc_wrapper(2)
        
        # Correct velocities and compute residuals
        res_u, res_v, res_p = correct_velocity(self.Var, self.VarOld, self.settings.dt, 
                                               self.fluid.rho, self.mesh.nx, self.mesh.ny, 
                                               self.mesh.dx, self.mesh.dy)
        self.residual[0] = res_u
        self.residual[1] = res_v
        self.residual[2] = res_p
        
        self._apply_bc_wrapper(0)
        self._apply_bc_wrapper(1)
        
        update_flux(self.Var, self.Ff, self.settings.dt, self.fluid.rho,
                   self.mesh.nx, self.mesh.ny, self.mesh.dx, self.mesh.dy)
    
    def _convergence_check(self, print_residuals: bool = False) -> Tuple[bool, np.ndarray]:
        """Check convergence based on residuals"""
        rms = np.zeros(self.nVar)
        for k in range(self.nVar):
            rms[k] = np.sqrt(self.residual[k] / (self.mesh.nx * self.mesh.ny))
            rms[k] = rms[k] / self.settings.dt
            if print_residuals:
                print(f"\t{rms[k]:.6e}", end="")
        
        if print_residuals:
            print()
        
        # Check for NaN or Inf in residuals
        if np.isnan(rms).any() or np.isinf(rms).any():
            print(f"\n❌ ERROR: NaN or Inf detected in residuals!")
            print(f"   U-residual: {rms[0]:.6e}, V-residual: {rms[1]:.6e}, P-residual: {rms[2]:.6e}")
            print(f"   This indicates solver instability or bad initial conditions.")
            raise ValueError("Solver failed: NaN/Inf in residuals")
        
        # Check convergence criteria
        converged = True
        if rms[0] > self.settings.convergence_criteria['u']:
            converged = False
        if rms[1] > self.settings.convergence_criteria['v']:
            converged = False
        if rms[2] > self.settings.convergence_criteria['p']:
            converged = False
        
        if not converged:
            copy_new_to_old(self.Var, self.VarOld, self.nVar, self.mesh.nx, self.mesh.ny)
        
        return converged, rms
    
    def _save_results(self, output_base_name: str):
        """Save all results including HDF5, plots, and data files"""
        # Create directory structure
        re_dir = os.path.dirname(output_base_name)
        if re_dir and not os.path.exists(re_dir):
            os.makedirs(re_dir)
        
        # Save to individual HDF5 file
        individual_h5 = f"{output_base_name}.h5"
        group_name = f"Re{int(self.fluid.Re)}_mesh{self.mesh.nx}x{self.mesh.ny}"
        self._save_results_hdf5(individual_h5, group_name)
        
        # Save to combined HDF5 file
        combined_h5 = "results/simulation_result_bfs.h5"
        self._save_results_hdf5(combined_h5, group_name)
        
        # Generate plots
        self._plot_centerlines(f"{output_base_name}_centerlines.png")
        self._plot_contours(f"{output_base_name}_contours.png")
        self._plot_convergence(f"{output_base_name}_convergence.png")
    
    def _save_results_hdf5(self, filename: str, group_name: str):
        """Save results to HDF5 file"""
        output_dir = os.path.dirname(filename)
        if output_dir and not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        with h5py.File(filename, 'a') as f:
            if group_name in f:
                del f[group_name]
            
            grp = f.create_group(group_name)
            
            # Set attributes (as per user requirements: case_name, bc_type with params, Re, nx, ny, lx, ly, step_height, h, Ub)
            grp.attrs["case_name"] = "BFS"
            grp.attrs["bc_type"] = f"BFS(step_height={self.step_height}, h={self.h}, Ub={self.Ub})"
            grp.attrs["reynolds_number"] = float(self.fluid.Re)
            grp.attrs["nx"] = int(self.mesh.nx)
            grp.attrs["ny"] = int(self.mesh.ny)
            grp.attrs["lx"] = float(self.mesh.lx)
            grp.attrs["ly"] = float(self.mesh.ly)
            grp.attrs["step_height"] = float(self.step_height)
            grp.attrs["h"] = float(self.h)
            grp.attrs["Ub"] = float(self.Ub)
            grp.attrs["total_points"] = int(self.mesh.nx * self.mesh.ny)
            
            # Create coordinate arrays
            x = np.linspace(0, self.mesh.lx, self.mesh.nx)
            y = np.linspace(0, self.mesh.ly, self.mesh.ny)
            X, Y = np.meshgrid(x, y)
            
            # Save datasets (flattened)
            grp.create_dataset("x", data=X.flatten())
            grp.create_dataset("y", data=Y.flatten())
            grp.create_dataset("u", data=self.Var[0, 1:-1, 1:-1].T.flatten())
            grp.create_dataset("v", data=self.Var[1, 1:-1, 1:-1].T.flatten())
            grp.create_dataset("p", data=self.Var[2, 1:-1, 1:-1].T.flatten())
    
    def _plot_centerlines(self, filename: str):
        """Plot centerline velocity profiles"""
        # Extract centerline data (at y = step_height + h/2)
        y_centerline_idx = int((self.step_height + self.h/2) / self.mesh.dy)
        y_centerline_idx = min(max(y_centerline_idx, 0), self.mesh.ny - 1)
        
        u_centerline = self.Var[0, 1:-1, y_centerline_idx + 1]
        v_centerline = self.Var[1, 1:-1, y_centerline_idx + 1]
        x = np.linspace(0, self.mesh.lx, self.mesh.nx)
        
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        ax1.plot(x, u_centerline, 'b-', linewidth=2)
        ax1.set_xlabel('X Position', fontsize=12)
        ax1.set_ylabel('U Velocity', fontsize=12)
        ax1.set_title(f'U velocity along horizontal centerline\n(Re={self.fluid.Re}, {self.mesh.nx}x{self.mesh.ny})', fontsize=11)
        ax1.grid(True, alpha=0.3)
        
        ax2.plot(x, v_centerline, 'r-', linewidth=2)
        ax2.set_xlabel('X Position', fontsize=12)
        ax2.set_ylabel('V Velocity', fontsize=12)
        ax2.set_title(f'V velocity along horizontal centerline\n(Re={self.fluid.Re}, {self.mesh.nx}x{self.mesh.ny})', fontsize=11)
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()
    
    def _plot_contours(self, filename: str):
        """Plot contour plots of all variables"""
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        
        # Create meshgrid for plotting
        x = np.linspace(0, self.mesh.lx, self.mesh.nx)
        y = np.linspace(0, self.mesh.ly, self.mesh.ny)
        X, Y = np.meshgrid(x, y)
        
        # U velocity contour
        im1 = axes[0, 0].contourf(X, Y, self.Var[0, 1:-1, 1:-1].T, levels=20, cmap='RdBu_r')
        axes[0, 0].set_title('U Velocity', fontsize=12)
        axes[0, 0].set_xlabel('X')
        axes[0, 0].set_ylabel('Y')
        axes[0, 0].axhline(y=self.step_height, color='black', linewidth=2, linestyle='--', alpha=0.5)
        plt.colorbar(im1, ax=axes[0, 0])
        
        # V velocity contour
        im2 = axes[0, 1].contourf(X, Y, self.Var[1, 1:-1, 1:-1].T, levels=20, cmap='RdBu_r')
        axes[0, 1].set_title('V Velocity', fontsize=12)
        axes[0, 1].set_xlabel('X')
        axes[0, 1].set_ylabel('Y')
        axes[0, 1].axhline(y=self.step_height, color='black', linewidth=2, linestyle='--', alpha=0.5)
        plt.colorbar(im2, ax=axes[0, 1])
        
        # Pressure contour
        im3 = axes[1, 0].contourf(X, Y, self.Var[2, 1:-1, 1:-1].T, levels=20, cmap='viridis')
        axes[1, 0].set_title('Pressure', fontsize=12)
        axes[1, 0].set_xlabel('X')
        axes[1, 0].set_ylabel('Y')
        axes[1, 0].axhline(y=self.step_height, color='black', linewidth=2, linestyle='--', alpha=0.5)
        plt.colorbar(im3, ax=axes[1, 0])
        
        # Velocity magnitude with streamlines
        u_mag = np.sqrt(self.Var[0, 1:-1, 1:-1]**2 + self.Var[1, 1:-1, 1:-1]**2)
        im4 = axes[1, 1].contourf(X, Y, u_mag.T, levels=20, cmap='plasma')
        axes[1, 1].set_title('Velocity Magnitude', fontsize=12)
        axes[1, 1].set_xlabel('X')
        axes[1, 1].set_ylabel('Y')
        axes[1, 1].axhline(y=self.step_height, color='white', linewidth=2, linestyle='--', alpha=0.7)
        
        # Add streamlines (subsample for clarity)
        skip = max(1, self.mesh.nx // 40)
        axes[1, 1].streamplot(X[::skip, ::skip], Y[::skip, ::skip], 
                             self.Var[0, 1:-1:skip, 1:-1:skip].T,
                             self.Var[1, 1:-1:skip, 1:-1:skip].T,
                             color='white', linewidth=0.5, density=1.0, arrowsize=0.8)
        plt.colorbar(im4, ax=axes[1, 1])
        
        plt.suptitle(f'BFS Flow (Re={self.fluid.Re}, {self.mesh.nx}x{self.mesh.ny})', fontsize=16, y=0.995)
        plt.tight_layout()
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()
    
    def _plot_convergence(self, filename: str):
        """Plot convergence history"""
        fig, ax = plt.subplots(1, 1, figsize=(10, 6))
        
        iterations = range(1, len(self.residual_history['u']) + 1)
        
        ax.semilogy(iterations, self.residual_history['u'], 'b-', label='U velocity', linewidth=2)
        ax.semilogy(iterations, self.residual_history['v'], 'r-', label='V velocity', linewidth=2)
        ax.semilogy(iterations, self.residual_history['p'], 'g-', label='Pressure', linewidth=2)
        
        ax.set_xlabel('Iteration', fontsize=12)
        ax.set_ylabel('RMS Residual', fontsize=12)
        ax.set_title(f'Convergence History (Re={self.fluid.Re}, {self.mesh.nx}x{self.mesh.ny})', fontsize=14)
        ax.legend(fontsize=11)
        ax.grid(True, alpha=0.3, which='both')
        
        plt.tight_layout()
        plt.savefig(filename, dpi=150, bbox_inches='tight')
        plt.close()

# Numba-compiled functions

@njit
def copy_new_to_old(Var, VarOld, nVar, Nx, Ny):
    for k in range(nVar):
        for i in range(Nx + 2):
            for j in range(Ny + 2):
                VarOld[k, i, j] = Var[k, i, j]

@njit
def apply_bc_configured(Var, k, Nx, Ny, bc_types, bc_values):
    """Apply boundary conditions based on configuration"""
    # Left and Right boundaries
    for j in range(1, Ny + 1):
        if bc_types[0] == 0:  # Dirichlet
            Var[k, 0, j] = 2 * bc_values[0] - Var[k, 1, j]
        else:  # Neumann
            Var[k, 0, j] = Var[k, 1, j]
        
        if bc_types[1] == 0:  # Dirichlet
            Var[k, Nx + 1, j] = 2 * bc_values[1] - Var[k, Nx, j]
        else:  # Neumann
            Var[k, Nx + 1, j] = Var[k, Nx, j]
    
    # Top and Bottom boundaries
    for i in range(1, Nx + 1):
        if bc_types[2] == 0:  # Dirichlet
            Var[k, i, Ny + 1] = 2 * bc_values[2] - Var[k, i, Ny]
        else:  # Neumann
            Var[k, i, Ny + 1] = Var[k, i, Ny]
        
        if bc_types[3] == 0:  # Dirichlet
            Var[k, i, 0] = 2 * bc_values[3] - Var[k, i, 1]
        else:  # Neumann
            Var[k, i, 0] = Var[k, i, 1]

@njit
def linear_interpolation(Var, Ff, Nx, Ny, dx, dy):
    for i in range(1, Nx + 1):
        for j in range(1, Ny + 1):
            Ff[0, i, j] = (Var[0, i, j] + Var[0, i + 1, j]) * dy * 0.5
            Ff[1, i, j] = (Var[1, i, j] + Var[1, i, j + 1]) * dx * 0.5
            Ff[2, i, j] = -(Var[0, i, j] + Var[0, i - 1, j]) * dy * 0.5
            Ff[3, i, j] = -(Var[1, i, j] + Var[1, i, j - 1]) * dx * 0.5

@njit
def simple_upwind(Var, Ff, k, i, j, volp):
    ue, uw, un, us = 0.0, 0.0, 0.0, 0.0
    sum_flux = 0.0
    
    if Ff[0, i, j] >= 0:
        ue = Var[k, i, j]
        sum_flux += Ff[0, i, j]
    else:
        ue = Var[k, i + 1, j]
    
    if Ff[2, i, j] >= 0:
        uw = Var[k, i, j]
        sum_flux += Ff[2, i, j]
    else:
        uw = Var[k, i - 1, j]
    
    if Ff[1, i, j] >= 0:
        un = Var[k, i, j]
        sum_flux += Ff[1, i, j]
    else:
        un = Var[k, i, j + 1]
    
    if Ff[3, i, j] >= 0:
        us = Var[k, i, j]
        sum_flux += Ff[3, i, j]
    else:
        us = Var[k, i, j - 1]
    
    Fc = ue * Ff[0, i, j] + uw * Ff[2, i, j] + un * Ff[1, i, j] + us * Ff[3, i, j]
    ap_c = sum_flux * volp
    
    return Fc, ap_c

@njit
def quick_scheme(Var, Ff, k, i, j, volp):
    ue, uw, un, us = 0.0, 0.0, 0.0, 0.0
    sum_flux = 0.0
    
    # East face
    if Ff[0, i, j] >= 0:
        ue = 0.75 * Var[k, i, j] + 0.375 * Var[k, i + 1, j] - 0.125 * Var[k, i - 1, j]
        sum_flux += 0.75 * Ff[0, i, j]
    else:
        ue = 0.75 * Var[k, i + 1, j] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i + 2, j]
        sum_flux += 0.375 * Ff[0, i, j]
    
    # West face
    if Ff[2, i, j] >= 0:
        uw = 0.75 * Var[k, i, j] + 0.375 * Var[k, i - 1, j] - 0.125 * Var[k, i + 1, j]
        sum_flux += 0.75 * Ff[2, i, j]
    else:
        uw = 0.75 * Var[k, i - 1, j] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i - 2, j]
        sum_flux += 0.375 * Ff[2, i, j]
    
    # North face
    if Ff[1, i, j] >= 0:
        un = 0.75 * Var[k, i, j] + 0.375 * Var[k, i, j + 1] - 0.125 * Var[k, i, j - 1]
        sum_flux += 0.75 * Ff[1, i, j]
    else:
        un = 0.75 * Var[k, i, j + 1] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i, j + 2]
        sum_flux += 0.375 * Ff[1, i, j]
    
    # South face
    if Ff[3, i, j] >= 0:
        us = 0.75 * Var[k, i, j] + 0.375 * Var[k, i, j - 1] - 0.125 * Var[k, i, j + 1]
        sum_flux += 0.75 * Ff[3, i, j]
    else:
        us = 0.75 * Var[k, i, j - 1] + 0.375 * Var[k, i, j] - 0.125 * Var[k, i, j - 2]
        sum_flux += 0.375 * Ff[3, i, j]
    
    Fc = ue * Ff[0, i, j] + uw * Ff[2, i, j] + un * Ff[1, i, j] + us * Ff[3, i, j]
    ap_c = sum_flux * volp
    
    return Fc, ap_c

@njit
def diffusive_flux(Var, k, i, j, dx, dy, volp):
    Fd = volp * ((Var[k, i + 1, j] - 2.0 * Var[k, i, j] + Var[k, i - 1, j]) / (dx * dx) +
                 (Var[k, i, j + 1] - 2.0 * Var[k, i, j] + Var[k, i, j - 1]) / (dy * dy))
    ap_d = -volp * (2.0 / (dx * dx) + 2.0 / (dy * dy))
    return Fd, ap_d

@njit
def update_flux(Var, Ff, dt, rho, Nx, Ny, dx, dy):
    for i in range(1, Nx + 1):
        for j in range(1, Ny + 1):
            Ff[0, i, j] += -dt / rho * (Var[2, i + 1, j] - Var[2, i, j]) * dy / dx
            Ff[1, i, j] += -dt / rho * (Var[2, i, j + 1] - Var[2, i, j]) * dx / dy
            Ff[2, i, j] += -dt / rho * (Var[2, i - 1, j] - Var[2, i, j]) * dy / dx
            Ff[3, i, j] += -dt / rho * (Var[2, i, j - 1] - Var[2, i, j]) * dx / dy

@njit(parallel=True)
def solve_momentum_quick(Var, VarOld, Ff, k, Nx, Ny, dx, dy, dt, nu, volp):
    # tolerance = 1e-6
    max_iter = 1000
    
    for iter in range(max_iter):
        # rms = 0.0
        for i in prange(1, Nx + 1):
            for j in range(1, Ny + 1):
                Fc, ap_c = quick_scheme(Var, Ff, k, i, j, volp)
                Fd, ap_d = diffusive_flux(Var, k, i, j, dx, dy, volp)
                
                # Implicit update formula: P = P_old + R / ap
                # R = Source - (FluxesIn - FluxesOut)
                # Unsteady term: volp/dt * (P - P_old)
                # Combined: P(volp/dt) + Fluxes = P_old(volp/dt) + Source
                
                R = -(volp / dt * (Var[k, i, j] - VarOld[k, i, j]) + Fc + (-nu) * Fd)
                ap = volp / dt + ap_c + (-nu) * ap_d
                
                Var[k, i, j] = Var[k, i, j] + R / ap
                # rms += R * R
        
        # rms = np.sqrt(rms / (Nx * Ny))
        # if rms < tolerance:
        #     break

@njit(parallel=True)
def solve_momentum_upwind(Var, VarOld, Ff, k, Nx, Ny, dx, dy, dt, nu, volp):
    # tolerance = 1e-6
    max_iter = 1000
    
    for iter in range(max_iter):
        # rms = 0.0
        for i in prange(1, Nx + 1):
            for j in range(1, Ny + 1):
                Fc, ap_c = simple_upwind(Var, Ff, k, i, j, volp)
                Fd, ap_d = diffusive_flux(Var, k, i, j, dx, dy, volp)
                
                R = -(volp / dt * (Var[k, i, j] - VarOld[k, i, j]) + Fc + (-nu) * Fd)
                ap = volp / dt + ap_c + (-nu) * ap_d
                
                Var[k, i, j] = Var[k, i, j] + R / ap
                # rms += R * R

@njit(parallel=True)
def solve_pressure(Var, Ff, Nx, Ny, dx, dy, dt, rho, volp):
    # tolerance = 1e-6
    max_iter = 1000
    k = 2  # Pressure
    
    for iter in range(max_iter):
        # rms = 0.0
        for i in prange(1, Nx + 1):
            for j in range(1, Ny + 1):
                Fd, ap_d = diffusive_flux(Var, k, i, j, dx, dy, volp)
                
                LHS = Fd
                RHS = rho / dt * (Ff[0, i, j] + Ff[1, i, j] + Ff[2, i, j] + Ff[3, i, j])
                R = RHS - LHS
                ap = ap_d
                
                Var[k, i, j] = Var[k, i, j] + R / ap
                # rms += R * R

@njit(parallel=True)
def correct_velocity(Var, VarOld, dt, rho, Nx, Ny, dx, dy):
    res_u = 0.0
    res_v = 0.0
    res_p = 0.0
    
    for i in prange(1, Nx + 1):
        for j in range(1, Ny + 1):
            # U velocity correction
            Var[0, i, j] = Var[0, i, j] - dt / rho * (Var[2, i + 1, j] - Var[2, i - 1, j]) / (2 * dx)
            # V velocity correction
            Var[1, i, j] = Var[1, i, j] - dt / rho * (Var[2, i, j + 1] - Var[2, i, j - 1]) / (2 * dy)
            
            # Calculate residuals
            res_u += (Var[0, i, j] - VarOld[0, i, j]) ** 2
            res_v += (Var[1, i, j] - VarOld[1, i, j]) ** 2
            res_p += (Var[2, i, j] - VarOld[2, i, j]) ** 2
    
    return res_u, res_v, res_p

@njit(parallel=True)
def under_relax_field(Var, VarOld, k, Nx, Ny, alpha):
    """Apply under-relaxation: phi = alpha * phi_new + (1-alpha) * phi_old"""
    for i in prange(1, Nx + 1):
        for j in range(1, Ny + 1):
            Var[k, i, j] = alpha * Var[k, i, j] + (1.0 - alpha) * VarOld[k, i, j]


## Simulation Parameters - Configure All Settings Here

In [ ]:
# ============================================================================
# SIMULATION PARAMETERS - MODIFY THESE AS NEEDED
# ============================================================================

# Reynolds Number Configuration
reynolds_numbers = [100]  # List of Reynolds numbers to simulate
                          # Examples: [100], [100, 200, 300], range(100, 801, 100)

# Mesh Size Configuration
mesh_sizes = [10,20,40,80,200,400]  # List of mesh resolutions (nx = ny)
                             # Common sizes: [10, 50, 100, 200, 400]

# BFS Geometry Parameters
step_height = 0.94 # Height of the backward-facing step
h = 1.0           # Channel height above the step
Ub = 1.0           # Bulk velocity at inlet
lx = 20.0          # Domain length in x-direction
ly = 1.94           # Domain height in y-direction (should be >= step_height + h)

# Convergence Criteria
convergence_criteria = {
    'u': 1e-6,
    'v': 1e-6,
    'p': 1e-6,
    'continuity': 1e-6
}

# Solver Settings
dt = 0.001              # Time step size
max_iterations = 120000 # Maximum number of iterations
scheme = 'QUICK'       # Numerical scheme: 'UPWIND' or 'QUICK'

# Under-Relaxation Factors (for stability)
relaxation_factors = {
    'u': 0.5,  # Relaxation for U velocity (0 < alpha <= 1)
    'v': 0.5,  # Relaxation for V velocity
    'p': 0.1   # Relaxation for pressure
}

# Fluid Properties
rho = 1.0  # Fluid density

# Output Directory
output_base_dir = "results"

print("="*70)
print("BFS GROUND TRUTH DATA GENERATION - PARAMETERS")
print("="*70)
print(f"Reynolds Numbers: {reynolds_numbers}")
print(f"Mesh Sizes: {mesh_sizes}")
print(f"BFS Geometry: step_height={step_height}, h={h}, Ub={Ub}")
print(f"Domain Size: lx={lx}, ly={ly}")
print(f"Time Step: dt={dt}")
print(f"Max Iterations: {max_iterations}")
print(f"Scheme: {scheme}")
print(f"Convergence Criteria: {convergence_criteria}")
print(f"Relaxation Factors: {relaxation_factors}")
print("="*70)

## Main Execution - Generate Ground Truth Data

In [ ]:
# ============================================================================
# MAIN EXECUTION LOOP
# ============================================================================

print("\n\n")
print("="*70)
print("STARTING BFS GROUND TRUTH DATA GENERATION")
print("="*70)

# Track overall progress
total_simulations = len(reynolds_numbers) * len(mesh_sizes)
current_simulation = 0

for Re in reynolds_numbers:
    # Create directory for this Reynolds number
    re_dir = os.path.join(output_base_dir, f"Re{Re}")
    os.makedirs(re_dir, exist_ok=True)
    
    print(f"\n{'='*70}")
    print(f"REYNOLDS NUMBER: Re = {Re}")
    print(f"{'='*70}")
    
    for size in mesh_sizes:
        current_simulation += 1
        nx = size
        ny = size
        
        print(f"\n[{current_simulation}/{total_simulations}] Processing: Re={Re}, Mesh={nx}x{ny}")
        print("-" * 70)
        
        # Create mesh-specific subdirectory
        mesh_dir = os.path.join(re_dir, f"mesh_{nx}x{ny}")
        os.makedirs(mesh_dir, exist_ok=True)
        
        # Create mesh parameters
        mesh = MeshParameters(nx=nx, ny=ny, lx=lx, ly=ly)
        
        # Create fluid properties
        fluid = FluidProperties(Re=Re, rho=rho)
        
        # Create solver settings
        solver_settings = SolverSettings(
            dt=dt,
            max_iterations=max_iterations,
            convergence_criteria=convergence_criteria,
            scheme=scheme,
            relaxation_factors=relaxation_factors
        )
        
        # Create boundary conditions (default BFS)
        bc = BoundaryConditions()
        
        # Output filename (inside mesh-specific subdirectory)
        output_name = os.path.join(mesh_dir, f"bfs_Re{Re}_mesh{nx}x{ny}")
        
        try:
            # Create and run solver
            solver = CFDSolver(
                mesh=mesh,
                fluid=fluid,
                solver_settings=solver_settings,
                bc=bc,
                step_height=step_height,
                h=h,
                Ub=Ub
            )
            
            iterations, elapsed_time = solver.solve(output_name, verbose=True)
            
            print(f"✓ Converged in {iterations} iterations ({elapsed_time:.2f} seconds)")
            print(f"✓ Saved to: {output_name}.h5")
            
        except Exception as e:
            print(f"✗ ERROR for Re={Re}, mesh={nx}x{ny}: {e}")
            import traceback
            traceback.print_exc()

print("\n\n")
print("="*70)
print("ALL SIMULATIONS COMPLETED!")
print("="*70)
print(f"Total simulations run: {total_simulations}")
print(f"Results saved in: {output_base_dir}/")
print(f"Structure: {output_base_dir}/Re*/mesh_*x*/bfs_Re*_mesh*x* (.h5, .png files)")
print(f"Combined file: {output_base_dir}/simulation_result_bfs.h5")
print("="*70)